In [1]:
import numpy as np
import librosa as lb
import HMMOLTW
from pathlib import Path
import numpy as np
import importlib
import IterativeTrainHMM
import matplotlib.pyplot as plt
import Constants
from scipy.special import logsumexp

importlib.reload(IterativeTrainHMM)
importlib.reload(HMMOLTW)

<module 'HMMOLTW' from 'c:\\Users\\morgp\\Documents\\.HMC\\.Year 3\\E207\\Python\\Project\\E207DTWHMM\\HMMOLTW.py'>

In [2]:
A = np.random.rand(5,5)
print(A)
B = np.max(A, axis = 0)
print(B)

[[0.13533307 0.46388614 0.42786629 0.60211414 0.5899361 ]
 [0.71230434 0.13816352 0.86574467 0.59036465 0.29419144]
 [0.15890193 0.15303834 0.18862756 0.5484653  0.71589135]
 [0.27251128 0.54325449 0.31669429 0.50041183 0.70343266]
 [0.2259714  0.24379179 0.0459835  0.48040711 0.9856961 ]]
[0.71230434 0.54325449 0.86574467 0.60211414 0.9856961 ]


In [3]:
PIECE_ID = "Chopin_Op030No2"
DATA_DIR = Path("data/wav_22050_mono") / PIECE_ID
TRAIN_FRACTION = 0.70
TEST_RECORDING_COUNT = 3

recording_paths = sorted(DATA_DIR.glob("*.wav"))
reference_path = recording_paths[0]

rng = np.random.default_rng(207)
query_pool = np.array(recording_paths[1:], dtype=object)
shuffled_pool = query_pool[rng.permutation(len(query_pool))]
train_count = int(np.ceil(TRAIN_FRACTION * len(recording_paths))) - 1
train_paths = sorted(shuffled_pool[:train_count])
heldout_paths = sorted(shuffled_pool[train_count:])
test_paths = heldout_paths[:TEST_RECORDING_COUNT]

print(f"reference: {reference_path.name}")
print(f"total recordings: {len(recording_paths)}")
print(f"training recordings: {1 + len(train_paths)} including the reference ({(1 + len(train_paths)) / len(recording_paths):.1%})")
print(f"held-out recordings: {len(heldout_paths)}")
print("test queries:")
for path in test_paths:
    print(f"  {path.name}")

reference: Chopin_Op030No2_Ashkenazy-1981_pid9058-19.wav
total recordings: 34
training recordings: 24 including the reference (70.6%)
held-out recordings: 10
test queries:
  Chopin_Op030No2_Chiu-1999_pid9048-19.wav
  Chopin_Op030No2_Cortot-1951_pid9066-19.wav
  Chopin_Op030No2_Fiorentino-1961_pid9065-14.wav


In [4]:
C = np.copy(B)
C[-1] = -np.inf
np.isfinite(C)

array([ True,  True,  True,  True, False])

In [5]:
TestFrom = np.array([1, 2, 3])
TestTo = np.array([2, 3, 4])

D = np.ix_(TestFrom, TestTo)
print(A)
print(A[D])

[[0.13533307 0.46388614 0.42786629 0.60211414 0.5899361 ]
 [0.71230434 0.13816352 0.86574467 0.59036465 0.29419144]
 [0.15890193 0.15303834 0.18862756 0.5484653  0.71589135]
 [0.27251128 0.54325449 0.31669429 0.50041183 0.70343266]
 [0.2259714  0.24379179 0.0459835  0.48040711 0.9856961 ]]
[[0.86574467 0.59036465 0.29419144]
 [0.18862756 0.5484653  0.71589135]
 [0.31669429 0.50041183 0.70343266]]


In [6]:
print(B)
print(A)

B[:, np.newaxis] + A

[0.71230434 0.54325449 0.86574467 0.60211414 0.9856961 ]
[[0.13533307 0.46388614 0.42786629 0.60211414 0.5899361 ]
 [0.71230434 0.13816352 0.86574467 0.59036465 0.29419144]
 [0.15890193 0.15303834 0.18862756 0.5484653  0.71589135]
 [0.27251128 0.54325449 0.31669429 0.50041183 0.70343266]
 [0.2259714  0.24379179 0.0459835  0.48040711 0.9856961 ]]


array([[0.84763741, 1.17619048, 1.14017063, 1.31441848, 1.30224045],
       [1.25555883, 0.68141801, 1.40899916, 1.13361914, 0.83744592],
       [1.0246466 , 1.018783  , 1.05437223, 1.41420997, 1.58163602],
       [0.87462542, 1.14536863, 0.91880843, 1.10252597, 1.3055468 ],
       [1.2116675 , 1.22948789, 1.0316796 , 1.46610321, 1.9713922 ]])

In [7]:
E = np.full(5, -np.inf)
E[3] = 4

print(logsumexp(E))

4.0


In [8]:
QueryFrameSequence =     np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2], dtype=np.int32)
ReferenceFrameSequence = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.int32)

QueryIncrements = np.diff(QueryFrameSequence) == 1
ToStates = ReferenceFrameSequence[1:][QueryIncrements]
FromStates = np.roll(ToStates, 1)
FromStates[0] = ReferenceFrameSequence[0]
# FromStates = ReferenceFrameSequence[:-1][QueryIncrements]

MaxState = max(ReferenceFrameSequence + 1)
A = np.zeros((MaxState, MaxState))

print(QueryFrameSequence)
print(ReferenceFrameSequence)

print(FromStates)
print(ToStates)

np.add.at(A, (FromStates, ToStates), 1)
print(A)


[0 0 0 0 1 1 1 1 2 2 2]
[ 0  1  2  3  4  5  6  7  8  9 10]
[0 4]
[4 8]
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [9]:
PossiblePreviousStates = [3, 4, 5]
CurrentPossibleStates= [1, 2, 3]

print(A)
print(A[np.ix_(PossiblePreviousStates, CurrentPossibleStates)])

[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
